# Task 3: Image inference với SDXL (Kaggle)

Notebook này sinh 4 ảnh thật cho storyboard mẫu bằng `SDXLImageGenerator`, rồi ghép truyện tranh 2×2 qua pipeline dùng chung của nhóm (`src.factory.build_pipeline`).

**Yêu cầu khi chạy**
- Accelerator: GPU T4. Kaggle cấp T4 x2, notebook chỉ dùng một GPU nhờ `CUDA_VISIBLE_DEVICES=0`.
- Internet: bật (tải mã nguồn từ GitHub và model từ Hugging Face).
- Chạy từ đầu (Run All) trong phiên mới. Mỗi phiên tải lại khoảng 6,5 GiB model; tổng thời gian khoảng 3 phút.

**Cố định để tái lập**
- Mã nguồn: commit `af9ada647266478ee331a4dc799e1354a9df3b15`.
- Model: `Lykon/dreamshaper-xl-v2-turbo`, revision `f70ad5bce436c640d23fc02cd0a6ccc787280b6e`.
- Lần chạy chính thức: Kaggle notebook `comic-gen-03-image-inference` (Private), Version 1. Chi tiết và số đo: `docs/image_generation.md`.

**Kết quả** nằm trong `/kaggle/working` (`run_task03/`, `task03_run.zip`, `env_report.json`). Không đưa ảnh, ZIP hay model vào Git; mỗi lần chạy có `SHA256SUMS` để kiểm tra toàn vẹn.

## 1. Kiểm tra môi trường (GPU, thư viện)

In [ ]:
import os

# Chỉ dùng một T4 (phải đặt trước khi import torch). Nếu đã import torch rồi thì Restart session.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import importlib.metadata as md
import platform
import shutil
import sys
import traceback

import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}, {p.total_memory / 2**30:.2f} GiB, compute capability {p.major}.{p.minor}")

print()
for pkg in ["diffusers", "transformers", "accelerate", "safetensors", "huggingface_hub", "Pillow", "numpy"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg}: KHONG CO")

print()
for path in ["/kaggle/working", os.path.expanduser("~")]:
    free = shutil.disk_usage(path).free / 2**30
    print(f"Dung luong trong tai {path}: {free:.1f} GiB")

print()
try:
    from diffusers import AutoPipelineForText2Image, DPMSolverMultistepScheduler  # noqa: F401
    print("Import diffusers (AutoPipelineForText2Image, DPMSolverMultistepScheduler): OK")
except Exception:
    print("Import diffusers: LOI")
    traceback.print_exc()

## 2. Thông tin model trên Hugging Face (không tải weights)

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "Lykon/dreamshaper-xl-v2-turbo"
info = HfApi().model_info(REPO_ID, files_metadata=True)

print("repo:", REPO_ID)
print("sha (revision de pin):", info.sha)
print("last_modified:", info.last_modified)
print("gated:", getattr(info, "gated", None), "| private:", getattr(info, "private", None))
card = getattr(info, "card_data", None)
print("license (card):", getattr(card, "license", None))
print()

files = sorted(info.siblings, key=lambda s: s.rfilename)
total_fp16 = 0
for s in files:
    size_mib = (s.size or 0) / 2**20
    marker = "  <- fp16" if ".fp16." in s.rfilename else ""
    print(f"{size_mib:10.1f} MiB  {s.rfilename}{marker}")
    if ".fp16." in s.rfilename:
        total_fp16 += s.size or 0

print()
print(f"Tong cac file .fp16.*: {total_fp16 / 2**30:.2f} GiB")
print("Co file fp16 de dung variant='fp16':", total_fp16 > 0)

## 3. Lưu báo cáo môi trường (`env_report.json`)

In [ ]:
import json
import time

report = {
    "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_build": torch.version.cuda,
    "gpus": [
        {"name": torch.cuda.get_device_properties(i).name,
         "total_gib": round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2)}
        for i in range(torch.cuda.device_count())
    ],
    "packages": {},
    "model": {"repo_id": REPO_ID, "sha": info.sha, "fp16_total_gib": round(total_fp16 / 2**30, 2)},
}
for pkg in ["diffusers", "transformers", "accelerate", "safetensors", "huggingface_hub", "Pillow", "numpy"]:
    try:
        report["packages"][pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        report["packages"][pkg] = None

with open("/kaggle/working/env_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2, ensure_ascii=False))

## 4. Lấy đúng mã nguồn (commit cố định)

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # một T4; phải đặt trước khi import torch

import importlib.metadata as md
import subprocess
import sys

REPO_URL = "https://github.com/trannam1084/comic-generation.git"
SOURCE_COMMIT = "af9ada647266478ee331a4dc799e1354a9df3b15"
WORKDIR = "/kaggle/working/comic-generation"

GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT="0")  # không bao giờ chờ nhập mật khẩu


def git(*args):
    done = subprocess.run(["git", *args], capture_output=True, text=True, env=GIT_ENV)
    if done.returncode != 0:
        raise RuntimeError(f"git {args[0]} that bai (ma {done.returncode}): {done.stderr.strip()[:300]}")
    return done.stdout.strip()


if not os.path.isdir(WORKDIR):
    git("clone", "--no-checkout", REPO_URL, WORKDIR)
git("-C", WORKDIR, "checkout", "--detach", SOURCE_COMMIT)
head = git("-C", WORKDIR, "rev-parse", "HEAD")
assert head == SOURCE_COMMIT, f"HEAD {head} khac commit mong doi"
print("HEAD:", head)
print("Thu muc sach (khong co thay doi cuc bo):", git("-C", WORKDIR, "status", "--porcelain") == "")

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)

for package in ["pydantic", "Pillow", "PyYAML", "torch", "diffusers", "transformers", "huggingface_hub"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: KHONG CO")

## 5. Nạp `SDXLImageGenerator` và sinh cảnh 1

In [ ]:
import json
import logging
import time
from pathlib import Path

import numpy as np
from IPython.display import display

from src.image_generator.prompt_builder import PromptBuilder
from src.image_generator.schemas import Storyboard as LegacyStoryboard
from src.image_generator.sdxl_generator import SDXLConfig, SDXLImageGenerator

logging.basicConfig(level=logging.INFO, format="%(message)s")

STYLE_E0 = "children's storybook illustration, clean line art, vibrant colors, single scene"
NEG_E0 = ("text, letters, watermark, signature, blurry, low quality, deformed, extra limbs, "
          "multiple panels, collage, split screen, comic page, frame, border, grid")

storyboard_data = json.loads(Path("examples/storyboard.json").read_text(encoding="utf-8"))
prompts = PromptBuilder(style=STYLE_E0, negative_prompt=NEG_E0).build(LegacyStoryboard.from_dict(storyboard_data))

generator = SDXLImageGenerator(SDXLConfig())  # mặc định: 6 bước, guidance 2.0, 1024x1024, một GPU
print("Prompt cảnh 1:", prompts[0].prompt)

first = generator.generate(prompts[0], seed=43)  # model được nạp ở lần gọi này; lỗi nếu có sẽ nổi lên nguyên vẹn
print(f"Nạp model (gồm tải): {generator.model_load_sec:.1f}s | sinh ảnh: {first.elapsed_sec:.1f}s | "
      f"seed {first.seed} | backend {first.backend}")
print("scheduler:", first.params["scheduler"])
print("phiên bản:", first.params["versions"])
gray = np.asarray(first.image.convert("L"), dtype=np.float32)
print(f"Kích thước {first.image.size}, trung bình {gray.mean():.1f}, độ lệch chuẩn {gray.std():.1f}")
display(first.image.resize((512, 512)))

## 6. Sinh 4 ảnh qua pipeline dùng chung và lưu kết quả

In [ ]:
import hashlib
import shutil

import torch

from src.configuration import PipelineConfig
from src.factory import build_pipeline
from src.storyboard.compat import LegacyPromptBuilderAdapter

OUT = Path("/kaggle/working/run_task03")
if OUT.exists():
    raise RuntimeError("Thư mục kết quả đã tồn tại: xóa hoặc đổi tên trước khi chạy để không ghi đè.")
(OUT / "raw").mkdir(parents=True)
(OUT / "comic").mkdir()

config = PipelineConfig(output_dir=OUT / "comic")
pipeline = build_pipeline(
    config,
    image_backend=generator,  # dùng lại model đã nạp ở Cell 2
    prompt_stage=LegacyPromptBuilderAdapter(PromptBuilder(style=STYLE_E0, negative_prompt=NEG_E0)),
)

start = time.perf_counter()
result = pipeline.generate(storyboard_data["summary"])
total_sec = time.perf_counter() - start

for generated in result.images:
    generated.image.save(OUT / "raw" / f"scene_{generated.scene_index}.png")
    print(f"cảnh {generated.scene_index}: seed={generated.seed}, {generated.elapsed_sec:.1f}s, backend={generated.backend}")

peak_gib = torch.cuda.max_memory_allocated() / 2**30
print(f"Tổng thời gian pipeline: {total_sec:.1f}s | VRAM đỉnh: {peak_gib:.2f} GiB")
print("comic:", result.comic_path)
print("Cảnh 1 khớp từng pixel với lần sinh riêng lẻ ở Cell 2 (cùng seed 43):",
      bool(np.array_equal(np.asarray(first.image), np.asarray(result.images[0].image))))

captions = {scene["index"]: scene["caption_vi"] for scene in storyboard_data["scenes"]}
metadata = {
    "source_commit": SOURCE_COMMIT,
    "model": {"repo_id": generator.config.repo_id, "revision": generator.config.revision},
    "prompt_style": STYLE_E0,
    "negative_prompt": NEG_E0,
    "pipeline_base_seed": config.base_seed,
    "accelerator": torch.cuda.get_device_name(0),
    "model_load_sec": round(generator.model_load_sec, 2),
    "pipeline_total_sec": round(total_sec, 2),
    "peak_vram_gib": round(peak_gib, 2),
    "scenes": [
        {
            "scene_index": generated.scene_index,
            "caption_vi": captions[generated.scene_index],
            "seed": generated.seed,
            "backend": generated.backend,
            "elapsed_sec": round(generated.elapsed_sec, 2),
            "prompt": prompt.prompt,
            "negative_prompt": prompt.negative_prompt,
            "params": generated.params,
        }
        for generated, prompt in zip(result.images, result.prompts)
    ],
}
(OUT / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


lines = [f"{sha256(p)}  {p.relative_to(OUT).as_posix()}" for p in sorted(OUT.rglob("*")) if p.is_file()]
(OUT / "SHA256SUMS").write_text("\n".join(lines) + "\n", encoding="utf-8")

archive = Path(shutil.make_archive("/kaggle/working/task03_run", "zip", OUT))
print("ZIP:", archive, f"{archive.stat().st_size / 2**20:.1f} MiB", "| sha256:", sha256(archive))
print(*lines, sep="\n")

## 7. Xem truyện tranh 2×2

In [ ]:
from PIL import Image

display(Image.open(result.comic_path))